# manual-chain-forward-and-back — worked example 3: Manual Forward and Backward Through cube → tanh

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `manual-chain-forward-and-back`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The chain `c = tanh(a³)` applies a polynomial transformation followed by a bounded nonlinearity. The backward pass demonstrates that `tanh_back` uses the cached output (`out`), while `cube_back` uses the cached input. Together they illustrate the two caching strategies in a single example — mirroring the pattern you will see again when implementing the ARENA autograd dispatcher.

## Worked solution

**Step 1 — forward pass: a → b → c.**
We compute `b = a³` (cube), then `c = tanh(b)`. Both intermediates are cached for the backward pass.

**Step 2 — tanh_back: compute dL/db.**
The derivative of tanh is `1 - tanh²(b) = 1 - out²`. So `dL/db = dL/dc * (1 - c²)`. We use the cached output `c`.

**Step 3 — cube_back: compute dL/da.**
The derivative of `a³` is `3a²`. So `dL/da = dL/db * 3 * a²`. We use the original input `a`.

**Step 4 — verify against autograd.**
The autograd path computes the same chain with gradient tracking and confirms numerical agreement.

In [ ]:
import torch as t

def tanh_back(grad_out, out, x):
    """d/dx tanh(x) = 1 - tanh^2(x) = 1 - out^2; uses cached output"""
    return grad_out * (1.0 - out ** 2)

def cube_back(grad_out, out, x):
    """d/dx x^3 = 3x^2; uses input x"""
    return grad_out * 3.0 * (x ** 2)

def manual_cube_tanh_chain(a, dL_dc):
    # Forward: a -> b -> c
    b = a ** 3           # b = a^3
    c = t.tanh(b)        # c = tanh(b)
    # Backward (reverse order)
    dL_db = tanh_back(dL_dc, c, b)   # uses cached output c
    dL_da = cube_back(dL_db, b, a)   # uses input a
    return b, c, dL_db, dL_da

# Run it
t.manual_seed(23)
a_val = t.tensor([-0.5, 0.3, 1.0, -1.2])
dL_dc_val = t.ones(4)

b, c, dL_db, dL_da = manual_cube_tanh_chain(a_val, dL_dc_val)
print(f"a            = {a_val}")
print(f"b = a^3      = {b.round(decimals=4)}")
print(f"c = tanh(b)  = {c.round(decimals=4)}")
print(f"dL/db        = {dL_db.round(decimals=4)}")
print(f"dL/da (manual)   = {dL_da.round(decimals=4)}")

# Verify with autograd
a_ag = a_val.clone().requires_grad_(True)
c_ag = t.tanh(a_ag ** 3)
loss = (c_ag * dL_dc_val).sum()
loss.backward()
print(f"dL/da (autograd) = {a_ag.grad.round(decimals=4)}")
print(f"Match: {t.allclose(dL_da, a_ag.grad, atol=1e-5)}")